In [ ]:

import openpyxl

# Function to load the workbook and get the sheet
def load_sheet(file_path, sheet_name):
    wb = openpyxl.load_workbook(file_path, data_only=True)  # data_only=True to load values not formulas
    return wb[sheet_name]

# Function to display column and row labels
def display_labels(sheet):
    columns = [sheet.cell(row=1, column=j).value for j in range(1, sheet.max_column + 1) if sheet.cell(row=1, column=j).value]
    rows = [sheet.cell(row=i, column=1).value for i in range(1, sheet.max_row + 1) if sheet.cell(row=i, column=1).value]

    print("Column Labels:")
    print(", ".join(columns))
    print("Row Labels:")
    print(", ".join(rows))

# Function to find cell value based on user inputs for row and column labels
def find_cell_value(sheet, row_label, column_label):
    row_index = None
    column_index = None

    # Normalize input for comparison
    row_label = row_label.strip().lower()
    column_label = column_label.strip().lower()

    # Search for the row index by checking each cell in the first column
    for i in range(1, sheet.max_row + 1):
        cell_value = sheet.cell(row=i, column=1).value
        if cell_value and str(cell_value).strip().lower() == row_label:
            row_index = i
            break

    # Search for the column index by checking each cell in the first row
    for j in range(1, sheet.max_column + 1):
        cell_value = sheet.cell(row=1, column=j).value
        if cell_value and str(cell_value).strip().lower() == column_label:
            column_index = j
            break

    # Retrieve the value at the intersection
    if row_index and column_index:
        result_value = sheet.cell(row=row_index, column=column_index).value
        return result_value if result_value is not None else "Empty cell"
    else:
        return f"Cell not found. Row index: {row_index}, Column index: {column_index}"

# Function to find applicable regulations based on asset name
def find_applicable_regulations(asset_name):
    asset_regulations = {
        "name": "GDPR, CCPA (if used for profiling)",
        "address": "GDPR, CCPA, PIPEDA",
        "blood type": "HIPAA, GDPR, Health Information Acts",
        "diseases": "HIPAA, GDPR, Health Information Acts",
        "age": "Age Discrimination Acts, GDPR, CCPA",
        "bank account number": "SOX, GLBA, PSD2",
        "passwords": "GDPR, CCPA, NIST Cybersecurity Framework",
        "credit/debit card details": "PCI-DSS, GDPR, CCPA",
        "email id": "GDPR, CCPA, CAN-SPAM Act",
        "bill/subscription payments": "SOX, GLBA, GDPR",
        "loan payments": "GLBA, SOX, Consumer Credit Acts",
        "transactions": "SOX, GLBA, PSD2",
        "housing address / location": "GDPR, CCPA, PIPEDA",
        "family member": "GDPR, CCPA, Family Rights Acts",
        "social security number": "GDPR (if applicable), CCPA, SSA",
        "fitness": "HIPAA (if through employer), GDPR",
        "health": "HIPAA, GDPR, Health Information Acts",
        "food/drinks": "Food Safety Acts, GDPR (if personalized)",
        "fantasies": "GDPR, CCPA",
        "vital/genes": "HIPAA, GINA, GDPR",
        "measurements": "HIPAA (if through employer), GDPR",
        "income": "IRS regulations, GDPR, CCPA",
        "budgets": "SOX, GDPR, CCPA",
        "loyalty": "GDPR, CCPA, Loyalty Program Acts",
        "employment": "Labor Laws, GDPR, CCPA",
        "skills": "GDPR, CCPA, Employment Acts",
        "achievements": "GDPR, CCPA, Employment Acts",
        "to do lists/tasks": "GDPR, CCPA, sector-specific regulations (if applicable)",
        "vehicles": "GDPR, CCPA, DMV regulations, EPA regulations (if in the U.S.)",
        "appliance": "Safety standards, energy standards",
        "nationality": "GDPR, CCPA, immigration laws, international employment laws",
        "smart appliance": "GDPR (for smart appliances)",
        "first name": "GDPR, CCPA (if used for profiling)",
        "last name": "GDPR, CCPA",
        "dob": "GDPR, CCPA, Age Discrimination Acts",
        "proof of id": "GDPR, CCPA, Identity Theft Protection Acts",
        "proof of address": "GDPR, CCPA, PIPEDA",
        "social security no.": "SSA, GDPR (if applicable), CCPA",
        "telephone number": "GDPR, CCPA, Telecommunication Regulations",
        "email address": "GDPR, CCPA, CAN-SPAM Act",
        "marital status": "GDPR, CCPA, Family Rights Acts",
        "employment status": "Labor Laws, GDPR, CCPA",
        "credit score": "FCRA, GDPR, CCPA",
        "taxpayer identification number": "IRS regulations, GDPR, CCPA",
        "passport number": "GDPR, CCPA, International Travel Regulations",
        "passport country of issuance": "GDPR, CCPA, International Travel Regulations",
        "alien identification card number": "Immigration Laws, GDPR, CCPA",
        "bank account pin number": "SOX, GLBA, PSD2",
        "outgoings": "IRS regulations, GDPR, CCPA",
        "personal real estate interest rate": "Truth in Lending Act, GDPR, CCPA",
        "home property price": "IRS regulations, GDPR, CCPA",
        "home property address": "GDPR, CCPA, PIPEDA",
        "home mortgage deposit %": "Truth in Lending Act, GDPR, CCPA",
        "home mortgage interest rate": "Truth in Lending Act, GDPR, CCPA",
        "insurance provider": "HIPAA, GDPR, CCPA, Insurance Regulations",
        "car insurance costs": "GDPR, CCPA, Vehicle Insurance Regulations",
        "price of car": "GDPR, CCPA, DMV regulations",
        "annual mileage": "GDPR, CCPA, EPA regulations (if in the U.S.)",
        "car use": "GDPR, CCPA, Vehicle Insurance Regulations",
        "interest rate on car loan": "Truth in Lending Act, GDPR, CCPA",
        "personal school fees": "GDPR, CCPA, Education Act",
        "course duration": "GDPR, CCPA, Education Act",
        "loan term": "Truth in Lending Act, GDPR, CCPA",
        "interest rate": "Truth in Lending Act, GDPR, CCPA",
        "loan purpose": "Truth in Lending Act, GDPR, CCPA",
        "current loan term": "Truth in Lending Act, GDPR, CCPA",
        "required loan term": "Truth in Lending Act, GDPR, CCPA",
        "current interest rate": "Truth in Lending Act, GDPR, CCPA",
        "investment account details": "SOX, GLBA, SEC Regulations, GDPR, CCPA",
        "investments": "SEC Regulations, GDPR, CCPA",
        "properties owned": "IRS regulations, GDPR, CCPA",
        "401(k)": "ERISA, IRS regulations, GDPR, CCPA",
        "403(b)": "ERISA, IRS regulations, GDPR, CCPA",
        "ira (individual retirement account)": "IRS regulations, GDPR, CCPA",
        "insurance": "HIPAA, GDPR, CCPA, Insurance Regulations",
        "tax record": "IRS regulations, GDPR, CCPA",
        "bank statements": "SOX, GLBA, GDPR, CCPA",
        "investments made": "SEC Regulations, GDPR, CCPA",
        "assets owned": "IRS regulations, GDPR, CCPA",
        "w-2": "IRS regulations, GDPR, CCPA",
        "1099 forms": "IRS regulations, GDPR, CCPA",
        "loan payment information": "Truth in Lending Act, GLBA, GDPR, CCPA",
        "health score": "HIPAA, GDPR, CCPA",
        "wills": "GDPR, CCPA, Estate and Trust Laws",
        "trusts": "GDPR, CCPA, Trust and Estate Laws",
        "real estate payments": "Truth in Lending Act, GDPR, CCPA",
        "loan payment history": "FCRA, Truth in Lending Act, GDPR, CCPA",
        "debt repayment plan": "Bankruptcy Act, Consumer Credit Acts, GDPR, CCPA",
        "bankruptcy filings": "Bankruptcy Act, GDPR, CCPA",
        "debt settlement agreements": "Consumer Credit Acts, GDPR, CCPA",
        "credit inquiries": "FCRA, GDPR, CCPA",
        "outstanding debt": "IRS regulations, GDPR, CCPA",
        "long term debt": "Truth in Lending Act, GDPR, CCPA",
        "short term debt": "Truth in Lending Act, GDPR, CCPA",
        "budgeting and spending pattern": "IRS regulations, GDPR, CCPA",
        "communication records": "GDPR, CCPA, Telecommunication Regulations",
        "investment strategies": "SEC Regulations, GDPR, CCPA",
        "past budgets": "IRS regulations, GDPR, CCPA",
        "personal spending trends": "GDPR, CCPA",
        "personal saving accounts": "IRS regulations, GDPR, CCPA",
        "personal medical condition": "HIPAA, GDPR, CCPA",
        "personal hardships": "GDPR, CCPA, Social Support Acts",
        "personal donations made": "IRS regulations, GDPR, CCPA",
        "personal contributions": "Campaign Finance Laws, GDPR, CCPA"
    }
    asset_name = asset_name.strip().lower()
    return asset_regulations.get(asset_name, "Regulations not found")

# Main function to execute the code
def main():
    file_path = input("Enter the path to your Excel file: ")
    sheet_name = input("Enter the sheet name: ")

    sheet = load_sheet(file_path, sheet_name)
    display_labels(sheet)

    row_label = input("Select a label from the first column for the row: ")
    column_label = input("Select a label from the first row for the column: ")

    value = find_cell_value(sheet, row_label, column_label)
    print(f"The value at '{row_label}' and '{column_label}' is: {value}")

    row_regulations = find_applicable_regulations(row_label)
    column_regulations = find_applicable_regulations(column_label)

    print(f"The applicable regulations for '{row_label}' are: {row_regulations}")
    print(f"The applicable regulations for '{column_label}' are: {column_regulations}")

# Run the main function
if __name__ == "__main__":
    main()


Enter the path to your Excel file: /content/Book4.xlsx


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np

def create_matrix(n, m):
    return np.arange(1, n * m + 1).reshape(n, m)

def get_average(matrix, row=None, column=None):
    if row is not None and column is not None:
        return float(matrix[row, column])  # Return the specific element
    elif row is not None:
        return np.mean(matrix[row, :])  # Average of the row
    elif column is not None:
        return np.mean(matrix[:, column])  # Average of the column
    else:
        return np.mean(matrix)  # Average of the entire matrix

# Define matrix dimensions
n = int(input("Enter the number of rows: "))
m = int(input("Enter the number of columns: "))
matrix = create_matrix(n, m)
print("Matrix:")
print(matrix)

# User input for rows and columns to average
row_input = input(f"Enter a row number to average (1-{n}), or leave blank to skip: ")
column_input = input(f"Enter a column number to average (1-{m}), or leave blank to skip: ")

row = int(row_input) - 1 if row_input.isdigit() else None
column = int(column_input) - 1 if column_input.isdigit() else None

# Validate the user input
if row is not None and (row < 0 or row >= n):
    print(f"Invalid row number. Please enter a number between 1 and {n}.")
elif column is not None and (column < 0 or column >= m):
    print(f"Invalid column number. Please enter a number between 1 and {m}.")
else:
    # Calculate and print the average based on user input
    if row is not None or column is not None:
        average = get_average(matrix, row=row, column=column)
        if row is not None and column is not None:
            print(f"Value at row {row+1}, column {column+1}: {average}")
        elif row is not None:
            print(f"Average of row {row+1}: {average}")
        elif column is not None:
            print(f"Average of column {column+1}: {average}")
    else:
        print("No row or column selected for averaging.")


In [ ]:
import pandas as pd

# Sample data in a dictionary format
data = {
    "Name": ["Alice", "Bob", "Charlie", "David"],
    "Address": ["123 Main St", "456 Elm St", None, "789 Oak St"],
    "DOB": ["1990-01-01", "1985-05-15", "1970-09-30", None],
    "Proof of ID": ["ID12345", "ID67890", None, "ID11111"],
    "Proof of Address": ["Proof1", None, "Proof3", "Proof4"],
    "Social Security No": ["SSN1", "SSN2", "SSN3", "SSN4"],
    "Telephone Number": ["1234567890", "0987654321", None, "1122334455"],
    "Email Address": ["alice@example.com", "bob@example.com", None, None],
    "Marital Status": ["Single", "Married", "Divorced", "Single"],
    "Employment Status": ["Employed", "Unemployed", "Employed", None],
    "Credit Score": [700, 650, None, 750],
    "Taxpayer Identification Number": ["TIN1", "TIN2", "TIN3", None],
    "Passport Number": ["P1", "P2", None, "P4"],
    "Passport Country of Issuance": ["Country1", "Country2", None, "Country4"],
    "Alien Identification Card Number": ["A1", "A2", None, "A4"],
    "Bank Account PIN Number": ["PIN1", "PIN2", None, "PIN4"]
}

# Convert dictionary to DataFrame
df = pd.DataFrame(data)

# Function to check for null values
def check_null_values(row):
    return row.isnull().sum()

# Apply function to each row
df['Null Count'] = df.apply(check_null_values, axis=1)

# Filter out rows with more than 2 null values
filtered_df = df[df['Null Count'] <= 2]

# Display the filtered DataFrame
print("Filtered Data (with <= 2 null values):")
print(filtered_df)

# Display the count of valid and invalid rows
print(f"\nTotal Data Sets: {len(df)}")
print(f"Valid Data Sets: {len(filtered_df)}")
print(f"Invalid Data Sets: {len(df) - len(filtered_df)}")


In [ ]:
import pandas as pd
import numpy as np

# Example data
data = {
    'personal': {'age': [25, 30, 22, 35], 'gender': [1, 0, 1, 0], 'marital_status': [0, 1, 0, 1]},
    'financial': {'income': [50000, 60000, 55000, 58000], 'debt': [5000, 10000, 7000, 12000], 'savings': [20000, 25000, 18000, 22000]},
    'health': {'weight': [70, 80, 60, 90], 'height': [175, 180, 165, 170], 'bmi': [22.9, 24.7, 22.0, 31.1]}
}

# Convert to DataFrame
personal_df = pd.DataFrame(data['personal'])
financial_df = pd.DataFrame(data['financial'])
health_df = pd.DataFrame(data['health'])

# Correlation calculation
personal_corr = personal_df.corr()
financial_corr = financial_df.corr()
health_corr = health_df.corr()

# Value assignment function
def calculate_value(correlation_matrix):
    value = correlation_matrix.abs().sum(axis=1)
    return value

personal_value = calculate_value(personal_corr)
financial_value = calculate_value(financial_corr)
health_value = calculate_value(health_corr)

# Combine values
values_df = pd.DataFrame({
    'personal_value': personal_value,
    'financial_value': financial_value,
    'health_value': health_value
})

# Pricing model
values_df['total_value'] = values_df.sum(axis=1)
values_df['price'] = (values_df['total_value'] / values_df['total_value'].max()) * 100  # Normalize prices

# Display the final pricing
print(values_df[['total_value', 'price']])


In [ ]:
 # Import necessary libraries
import pandas as pd

# Risk types as columns in the table
risk_types = [
    'Identifiability', 'Sensitivity', 'Confidentiality', 'Competitiveness', 'Reputation', 'Compliance', 'Financial',
    'Political', 'Diplomatic', 'Emotional', 'Commercial', 'Legal', 'Geopolitical', 'Military', 'Intelligence',
    'Social', 'Public welfare', 'Strategic', 'Operational', 'Environmental', 'Security', 'Health and safety'
]

# Risk matrix as per the uploaded table
risk_matrix = {
    'Name': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Low', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Low', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Address': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Medium', 'Diplomatic': 'Medium', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Medium', 'Military': 'Medium', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Medium', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Medium', 'Security': 'Medium', 'Health and safety': 'Medium'},
    'Blood Type': {'Identifiability': 'Low', 'Sensitivity': 'High', 'Confidentiality': 'Medium', 'Competitiveness': 'Low', 'Reputation': 'Medium', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Diseases': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Low', 'Reputation': 'High', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'High', 'Commercial': 'Low', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'High'},
    'Age': {'Identifiability': 'Medium', 'Sensitivity': 'Medium', 'Confidentiality': 'Low', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Low', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Bank Account Number': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Passwords': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'High', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Credit/Debit Card Details': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Email ID': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Bill/Subscription Payments': {'Identifiability': 'Low', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Loan Payments': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Transactions': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Housing Address / Location': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Family Member': {'Identifiability': 'Low', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Low', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Social Security Number': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Fitness': {'Identifiability': 'Low', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Health': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Low', 'Reputation': 'High', 'Compliance': 'Medium', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'High', 'Commercial': 'Low', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'High'},
    'Food/Drinks': {'Identifiability': 'Low', 'Sensitivity': 'Low', 'Confidentiality': 'Low', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Fantasies': {'Identifiability': 'Low', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Low', 'Reputation': 'High', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'High', 'Commercial': 'Low', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Vital/Genes': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'High', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'High', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'High'},
    'Measurements': {'Identifiability': 'Medium', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Income': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Budgets': {'Identifiability': 'Low', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Loyalty': {'Identifiability': 'Low', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Employment': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Skills': {'Identifiability': 'Low', 'Sensitivity': 'Low', 'Confidentiality': 'Low', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Achievements': {'Identifiability': 'Low', 'Sensitivity': 'Low', 'Confidentiality': 'Low', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'To Do Lists/Tasks': {'Identifiability': 'Low', 'Sensitivity': 'Low', 'Confidentiality': 'Low', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Vehicles': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Appliance': {'Identifiability': 'Low', 'Sensitivity': 'Low', 'Confidentiality': 'Low', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Low', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Nationality': {'Identifiability': 'Low', 'Sensitivity': 'Medium', 'Confidentiality': 'Low', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Low', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Smart Appliance': {'Identifiability': 'Low', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'First Name': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Low', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Low', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Last Name': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Low', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Low', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'DOB': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Low', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Low', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Medium'},
    'Proof of ID': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Medium'},
    'Proof of Address': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Medium'},
    'Social Security No.': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Telephone Number': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Email Address': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Marital Status': {'Identifiability': 'Medium', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Low', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Low', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Employment Status': {'Identifiability': 'Medium', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Credit Score': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Taxpayer Identification Number': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Passport Number': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'Low', 'Political': 'Medium', 'Diplomatic': 'High', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'High', 'Geopolitical': 'Medium', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Passport Country of Issuance': {'Identifiability': 'Medium', 'Sensitivity': 'High', 'Confidentiality': 'Medium', 'Competitiveness': 'Low', 'Reputation': 'Medium', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Medium', 'Diplomatic': 'High', 'Emotional': 'Medium', 'Commercial': 'Low', 'Legal': 'Medium', 'Geopolitical': 'Medium', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Alien Identification Card Number': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'High', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Bank Account PIN Number': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Income': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Outgoings': {'Identifiability': 'Low', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Personal Real Estate Interest Rate': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Low', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Home Property Price': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Home Property Address': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Home Mortgage Deposit %': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Home Mortgage Interest Rate': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Insurance Provider': {'Identifiability': 'Medium', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Car Insurance Costs': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Price of Car': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Annual Mileage': {'Identifiability': 'Medium', 'Sensitivity': 'Low', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Car Use': {'Identifiability': 'Low', 'Sensitivity': 'Low', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Interest Rate on Car Loan': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Personal School Fees': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Course Duration': {'Identifiability': 'Medium', 'Sensitivity': 'Low', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Loan Term': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Interest Rate': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Loan Purpose': {'Identifiability': 'Medium', 'Sensitivity': 'High', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Current Loan Term': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Required Loan Term': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'Medium', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Current Interest Rate': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Investment Account Details': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Investments': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'Medium', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Properties Owned': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    '401(k)': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    '403(b)': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'IRA (Individual Retirement Account)': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Insurance': {'Identifiability': 'Medium', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Tax Record': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Bank Statements': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Investments Made': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Assets Owned': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'W-2': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    '1099 Forms': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Loan Payment Information': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'Medium', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'High', 'Commercial': 'Low', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Medium', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'High'},
    'Health Score': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Wills': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Trusts': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'Medium', 'Compliance': 'Medium', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Real Estate Payments': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Loan Payment History': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Debt Repayment Plan': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Medium', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Bankruptcy Filings': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Debt Settlement Agreements': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'Medium', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Medium', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Medium', 'Environmental': 'Low', 'Security': 'Medium', 'Health and safety': 'Low'},
    'Credit Inquiries': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Outstanding Debt': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Long Term Debt': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Short Term Debt': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Budgeting and Spending Pattern': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Communication Records': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Investment Strategies': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Past Budgets': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Personal Spending Trends': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'High', 'Reputation': 'High', 'Compliance': 'High', 'Financial': 'High', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'High', 'Legal': 'High', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'High', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'High', 'Operational': 'High', 'Environmental': 'Low', 'Security': 'High', 'Health and safety': 'Low'},
    'Personal Saving Accounts': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Low', 'Reputation': 'High', 'Compliance': 'Medium', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'High', 'Commercial': 'Low', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Medium', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'High'},
    'Personal Medical Condition': {'Identifiability': 'High', 'Sensitivity': 'High', 'Confidentiality': 'High', 'Competitiveness': 'Medium', 'Reputation': 'High', 'Compliance': 'Medium', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'High', 'Commercial': 'Low', 'Legal': 'Medium', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Medium', 'Public welfare': 'Low', 'Strategic': 'Medium', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'High'},
    'Personal Hardships': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'High', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'},
    'Personal Donations Made': {'Identifiability': 'High', 'Sensitivity': 'Medium', 'Confidentiality': 'High', 'Competitiveness': 'Low', 'Reputation': 'Low', 'Compliance': 'Low', 'Financial': 'Low', 'Political': 'Low', 'Diplomatic': 'Low', 'Emotional': 'Medium', 'Commercial': 'Low', 'Legal': 'Low', 'Geopolitical': 'Low', 'Military': 'Low', 'Intelligence': 'Low', 'Social': 'Low', 'Public welfare': 'Low', 'Strategic': 'Low', 'Operational': 'Low', 'Environmental': 'Low', 'Security': 'Low', 'Health and safety': 'Low'}
    }

# Scoring for each level
score_mapping = {'High': 3, 'Medium': 2, 'Low': 1}


# Combine the Qubes into a dictionary for easy access
qubes_dict = {
    "Open Bank Account Qube": ['Name', 'Address', 'DOB', 'Proof of ID', 'Proof of Address', 'Social Security No.', 'Telephone Number', 'Email Address', 'Marital Status', 'Employment Status', 'Credit Score', 'Taxpayer identification number', 'Passport number', 'Alien identification card number', 'Bank Account PIN number'],
    "New Credit Card Qube": ['Name', 'Address', 'DOB', 'Proof of ID', 'Proof of Address', 'Social Security No.', 'Telephone Number', 'Email Address', 'Marital Status', 'Employment Status', 'Credit Score', 'Income', 'Outgoings', 'Interest Rate'],
    "Mortgage Application Qube": ['Name', 'Address', 'DOB', 'Proof of ID', 'Proof of Address', 'Social Security No.', 'Telephone Number', 'Email Address', 'Marital Status', 'Employment Status', 'Credit Score', 'Income', 'Outgoings', 'Property Price', 'Property Address', 'Deposit %', 'Interest Rate', 'Insurance Provider', 'Insurance Costs'],
    "Car Finance Qube": ['Name', 'Address', 'DOB', 'Proof of ID', 'Proof of Address', 'Social Security No.', 'Telephone Number', 'Email Address', 'Marital Status', 'Employment Status', 'Credit Score', 'Income', 'Outgoings', 'Price of Car', 'Annual Mileage', 'Car Use', 'Interest Rate', 'Insurance Provider'],
    "Student Loan Application Qube": ['Name', 'Address', 'DOB', 'Proof of ID', 'Proof of Address', 'Social Security No.', 'Telephone Number', 'Email Address', 'Marital Status', 'Employment Status', 'Credit Score', 'Income', 'Outgoings', 'School Fees', 'Course Duration', 'Loan Term', 'Interest Rate', 'Insurance Provider'],
    "Refinance Qube": ['Name', 'Address', 'DOB', 'Proof of ID', 'Proof of Address', 'Social Security No.', 'Telephone Number', 'Email Address', 'Marital Status', 'Employment Status', 'Credit Score', 'Income', 'Outgoings', 'Loan Purpose', 'Current Loan Term', 'Required Loan Term', 'Current Interest Rate', 'Insurance Provider'],
    "Investment Qube": ['Name', 'Email Address', 'Marital Status', 'Employment Status', 'Credit Score', 'Income', 'Outgoings', 'Risk Profile', 'Investment Goal', 'Residency', 'Acreddited or not', 'Restrictions', 'Don’t invest in List', 'Account Details'],
    "Retirement Plan Qube": ['Name', 'Address', 'DOB', 'Phone number', 'Identification number', 'Income', 'Tax record', 'Health Score', 'Trusts', 'Credit Score', '401(k)', '403(b)', 'IRA (individual retirement account)', 'Investment'],
    "Prepare Taxes Qube": ['Name', 'Address', 'DOB', 'Phone number', 'Identification number', 'Income', 'Tax record', 'Bank Statements', 'Credit Score', 'Employment Record', 'W-2', '1099 forms', 'Payment Information'],
    "Debt Managment Qube": ['Name', 'Address', 'DOB', 'Phone number', 'Identification number', 'Income', 'Tax record', 'Credit Score', 'Personal Loans', 'Student Loans', 'Mortage Loans', 'Debt repayment plan', 'Debt settlement agreements', 'credit inquiries', 'outstanding debt', 'long term debt', 'short term debt'],
    "Savnigs Plan Qube": ['Name', 'Address', 'DOB', 'Phone number', 'Identification number', 'Income', 'Tax record', 'Credit Score', 'Bank Account', 'Investment account', 'Past budgets', 'Budgeting and Spending pattern', 'Saving accounts', 'Investment Strategies'],
    "Budgeting Qube": ['Name', 'Address', 'DOB', 'Phone number', 'Identification number', 'Income', 'Tax record', 'Credit Score', 'Bank Account', 'Communication Records', 'Spending trends', 'Investment account', 'Real estates', 'Medical condition'],
    "Fund Raising Qube": ['Revenue', 'Expenses', 'Profit margins', 'Cashflow statements', 'Balance sheet', 'Income Statement', 'Financial Leverage ratio', 'PE', 'Return on investment', 'Return on asset', 'Ownership structures', 'Type of business'],
    "Business Loan Qube": ['Name', 'Address', 'DOB', 'Phone number', 'Identification number', 'Income', 'Tax record', 'Credit Score', 'Ownership structures', 'Return on investment', 'Business credit reports', 'Collateral offered', 'Loan approval decision', 'Funding disbursements', 'Repayment schedules'],
    "Staking Yield Qube": ['Name', 'Address', 'DOB', 'Phone number', 'Identification number', 'Income', 'Bank Account', 'Type of currency', 'Wallet Address', 'Transaction history', 'Experience', 'Yield Earnings', 'Data exchange information'],
    "Trading Currencies Qube": ['Name', 'Address', 'DOB', 'Phone number', 'Identification number', 'Income', 'Bank Account', 'Type of currency', 'Wallet Address', 'Transaction history', 'Experience', 'Social Status']
}

# Extract just the Qube names and label them as qubes
qubes = {"qubes": list(qubes_dict.keys())}

print(qubes)

# Function to calculate the score for a given qube
def calculate_risk_score(qubes, risk_matrix, risk_types, score_mapping):
    total_score = 0
    max_score = 0
    # Iterate over each qube provided
    for qube in qubes:
        if qube in qubes_dict:
            for record in qubes_dict[qube]:
                if record in risk_matrix:
                    for risk_type in risk_types:
                        risk_value = risk_matrix[record].get(risk_type, 'Low')  # Default to 'Low' if no risk type is provided
                        total_score += score_mapping.get(risk_value, 1)  # Default to Low if missing
                        max_score += 3  # Max score for each type is High (3)
        else:
            print(f"Qube '{qube}' not found.")

    percentage_score = (total_score / max_score) * 100 if max_score != 0 else 0
    return total_score, percentage_score

# Function to classify the risk level based on the score
def classify_risk(percentage_score):
    if percentage_score > 66:
        return 'High Risk'
    elif percentage_score > 33:
        return 'Medium Risk'
    else:
        return 'Low Risk'

# Function to accept user input for which Qubes to combine and handle errors
def get_user_input_qubes():
    available_qubes = list(qubes_dict.keys())
    print("Available Qubes: ", available_qubes)

    while True:
        user_input = input("Enter the Qubes you want to score, separated by commas (e.g., OpenBankAccountQube,NewCreditCardQube): ")

        # Split the input and strip whitespace, also handle capitalization
        selected_qubes = [qube.strip() for qube in user_input.split(',')]

        # Check for invalid Qube names
        invalid_qubes = [qube for qube in selected_qubes if qube not in available_qubes]

        if invalid_qubes:
            print(f"Invalid Qubes: {invalid_qubes}")
            print("Please try again.")
        else:
            return selected_qubes

# Get user input for Qubes to calculate combined risk score
selected_qubes = get_user_input_qubes()

# Calculate combined risk score for selected Qubes
combined_score, combined_percentage = calculate_risk_score(selected_qubes, risk_matrix, risk_types, score_mapping)

# Classify the risk level
combined_risk_level = classify_risk(combined_percentage)

# Display the combined results
print(f"Selected Qubes: {selected_qubes}")
print(f"Total Combined Score = {combined_score}")
print(f"Percentage Score = {combined_percentage:.2f}%")
print(f"Overall Risk Level = {combined_risk_level}")


In [ ]:
import pandas as pd
import numpy as np

# Step 1: Convert `risk_matrix` to DataFrame and map risk levels to scores
risk_df = pd.DataFrame(risk_matrix).T.replace(score_mapping)

# Step 2: Apply risk type weights
# Create a series from `risk_weights` with risk types as index
risk_weights_series = pd.Series(risk_weights)

# Apply risk weights to each column in `risk_df`
weighted_risk_df = risk_df * risk_weights_series

# Step 3: Calculate Qube Scores
def calculate_vectorized_risk_score(selected_qubes, weighted_risk_df, qubes_dict, qube_weights):
    total_weighted_score = 0
    max_weighted_score = 0

    for qube in selected_qubes:
        if qube in qubes_dict:
            qube_weight = qube_weights.get(qube, 1)  # Default weight if not specified

            # Filter the rows in `weighted_risk_df` corresponding to the records in this Qube
            records_in_qube = qubes_dict[qube]
            qube_scores = weighted_risk_df.loc[records_in_qube]

            # Sum up all scores and apply qube weight
            total_score = qube_scores.sum().sum() * qube_weight
            total_weighted_score += total_score

            # Calculate max score assuming max score for each risk type is 3
            max_score_per_record = 3 * risk_weights_series * qube_weight
            max_weighted_score += max_score_per_record.sum() * len(records_in_qube)

    # Calculate the percentage score
    percentage_score = (total_weighted_score / max_weighted_score) * 100 if max_weighted_score != 0 else 0
    return total_weighted_score, percentage_score

# Sample usage
selected_qubes = ["Open Bank Account Qube", "New Credit Card Qube"]
combined_score, combined_percentage = calculate_vectorized_risk_score(selected_qubes, weighted_risk_df, qubes_dict, qube_weights)
combined_risk_level = classify_risk(combined_percentage)

print(f"Selected Qubes: {selected_qubes}")
print(f"Total Combined Weighted Score = {combined_score}")
print(f"Percentage Weighted Score = {combined_percentage:.2f}%")
print(f"Overall Risk Level = {combined_risk_level}")
